In [1]:
from xmlrpc.server import SimpleXMLRPCServer
from xmlrpc.server import SimpleXMLRPCRequestHandler

import datetime
import os
import threading
import glob

class RequestHandler(SimpleXMLRPCRequestHandler):
    rpc_paths = ('/RPC2',)


server = SimpleXMLRPCServer(("localhost", 8018),
                            requestHandler=RequestHandler,
                            allow_none=True)
server.register_introspection_functions()


LOG_FILE = 'log.csv'
MAX_RECORDS = 20
lock = threading.Lock()  # чтобы два события не писались одновременно
#Считает, сколько строк в файле
def count_records():
    if not os.path.exists(LOG_FILE):
        return 0
    with open(LOG_FILE, 'r', encoding='utf-8') as f:
        return sum(1 for _ in f)


def rotate_log():
    if os.path.exists(LOG_FILE):
        stamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
        new_name = f'log_{stamp}.csv'
        os.rename(LOG_FILE, new_name)
        print(f"[ROTATE] log.csv -> {new_name}")


def ping_stats():
    return "stats_server_ok"

server.register_function(ping_stats, 'ping_stats')

#Возвращает список зарегистрированных методов
def list_stats_methods():
    return server.system_listMethods()

server.register_function(list_stats_methods, 'list_stats_methods')


def add_log(event_type):
    with lock:
        if count_records() >= MAX_RECORDS:
            rotate_log()
        with open(LOG_FILE, 'a', encoding='utf-8') as f:
            f.write(str(event_type) + ',' + datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S") + '\n')
    return True

server.register_function(add_log, 'add_log')


#Пункт 6
def get_log(event_type=None, dt_from=None, dt_to=None):
    """
    Читает ВСЕ файлы log*.csv (текущий + ротированные),
    фильтрует по типу и времени, возвращает единый отсортированный список.
    """
    rows = []

    # Все файлы log*.csv в текущей папке:
    # log.csv + log_YYYYMMDD_hhmmss.csv
    files = glob.glob('log*.csv')

    for filepath in files:
        with open(filepath, 'r', encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue

                parts = line.split(',', 1)
                if len(parts) != 2:
                    continue
                ev, ts = parts[0].strip(), parts[1].strip()

                # фильтр по типу события
                if event_type and ev != event_type:
                    continue

                try:
                    t = datetime.datetime.strptime(ts, "%Y-%m-%d %H:%M:%S")
                except ValueError:
                    continue

                # фильтр по времени
                if dt_from:
                    if t < datetime.datetime.strptime(dt_from, "%Y-%m-%d %H:%M:%S"):
                        continue
                if dt_to:
                    if t > datetime.datetime.strptime(dt_to, "%Y-%m-%d %H:%M:%S"):
                        continue

                rows.append((ev, ts))

    # сортируем по времени (от старых к новым)
    rows.sort(key=lambda r: r[1])

    return rows

server.register_function(get_log, 'get_log')
# def get_log(event_type=None, dt_from=None, dt_to=None):
#     rows = []
#     if not os.path.exists(LOG_FILE):
#         return rows

#     with open(LOG_FILE, 'r', encoding='utf-8') as f:
#         for line in f:
#             line = line.strip()
#             if not line:
#                 continue

#             parts = line.split(',', 1)
#             if len(parts) != 2:
#                 continue
#             ev, ts = parts[0].strip(), parts[1].strip()
#             # фильтр по типу события
#             if event_type and ev != event_type:
#                 continue

#             try:
#                 t = datetime.datetime.strptime(ts, "%Y-%m-%d %H:%M:%S")
#             except ValueError:
#                 continue

#             if dt_from:
#                 if t < datetime.datetime.strptime(dt_from, "%Y-%m-%d %H:%M:%S"):
#                     continue
#             if dt_to:
#                 if t > datetime.datetime.strptime(dt_to, "%Y-%m-%d %H:%M:%S"):
#                     continue

#             rows.append((ev, ts))

#     return rows

# server.register_function(get_log, 'get_log')
print("Listening on port 8018...")

server.serve_forever()

Listening on port 8018...


127.0.0.1 - - [19/Oct/2026 14:57:36] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [19/Oct/2026 14:57:36] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [19/Oct/2026 14:57:36] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [19/Oct/2026 14:57:36] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [19/Oct/2026 14:57:36] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [19/Oct/2026 14:57:36] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [19/Oct/2026 14:57:36] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [19/Oct/2026 14:57:36] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [19/Oct/2026 14:57:36] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [19/Oct/2026 14:57:36] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [19/Oct/2026 14:57:36] "POST /RPC2 HTTP/1.1" 200 -


[ROTATE] log.csv -> log_20261019_145736.csv


127.0.0.1 - - [19/Oct/2026 14:57:38] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [19/Oct/2026 14:57:38] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [19/Oct/2026 14:57:38] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [19/Oct/2026 14:57:38] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [19/Oct/2026 14:57:39] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [19/Oct/2026 14:57:40] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [19/Oct/2026 14:57:39] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [19/Oct/2026 14:57:39] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [19/Oct/2026 14:57:39] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [19/Oct/2026 14:57:40] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [19/Oct/2026 14:57:40] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [19/Oct/2026 14:57:40] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [19/Oct/2026 14:57:41] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [19/Oct/2026 14:57:41] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [19/Oct/2026 14:57:41] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [19/Oct/202

KeyboardInterrupt: 